<a href="https://colab.research.google.com/github/Ashonet/S-P100_ANALYSIS/blob/main/Stock_Analysis_All_Round.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stock Valuation Analysis
Five models: DDM · Graham · RIM · DCF · PEG  
Run each section independently. The shared core cell must be executed first.

| Universe | Stocks | Est. runtime |
|---|---|---|
| S&P 100 | ~100 | ~5 min |
| S&P 500 | ~500 | ~25 min |
| S&P 400 Mid Cap | ~400 | ~20 min |
| Russell 2000 | ~2000 | ~90 min |

## Setup
Run once per Colab session.

In [1]:
!pip install yfinance --upgrade -q
!pip install pandas numpy lxml html5lib beautifulsoup4 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 46.6 MB/s eta 0:00:00


## Shared Core
Run this cell before any universe section below. Contains all models, market-parameter derivation, and output formatting.

In [2]:
import concurrent.futures
import io
import logging
import time
import urllib.request
from dataclasses import dataclass
from typing import Optional

import numpy as np
import pandas as pd
import yfinance as yf

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
log = logging.getLogger(__name__)

ValuationResult = tuple[str, float]
GRAHAM_MULTIPLIER = 22.5
DCF_FORECAST_YEARS = 10



# Market parameters

@dataclass
class MarketParams:
    risk_free_rate:      float
    equity_risk_premium: float
    discount_rate:       float
    terminal_growth:     float
    peg_buy_threshold:   float
    peg_sell_threshold:  float


def _fetch_treasury_yield() -> float:
    """13-week T-bill yield from ^IRX. Falls back to 4.5 %."""
    try:
        hist = yf.Ticker("^IRX".history(period="5d"))
        if not hist.empty:
            return float(hist["Close"].iloc[-1]) / 100
    except Exception:
        pass
    log.warning("Could not fetch T-bill yield; using 4.5 %")
    return 0.045


def _fetch_equity_risk_premium(risk_free: float) -> float:
    """
    Blended ERP: 50 % Damodaran long-run historical (4.2 %%) +
    50 % trailing 10-yr implied (S&P 500 return - rf), clamped to [2 %, 6 %].
    """
    HISTORICAL_ERP = 0.042
    try:
        hist = yf.Ticker("^GSPC".history(period="10y")["Close"].dropna())
        if len(hist) > 252:
            years = len(hist) / 252
            trailing_erp = (hist.iloc[-1] / hist.iloc[0]) ** (1 / years) - 1 - risk_free
            return float(np.clip(0.5 * HISTORICAL_ERP + 0.5 * trailing_erp, 0.02, 0.06))
    except Exception:
        pass
    log.warning("Could not compute ERP; using 4.2 %")
    return HISTORICAL_ERP


def _fetch_terminal_growth() -> float:
    """10-yr breakeven inflation from FRED (T10YIE). Falls back to 2.5 %."""
    try:
        url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=T10YIE"
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=10) as resp:
            rows = resp.read().decode().strip().splitlines()
        for row in reversed(rows[1:]):
            parts = row.split(",")
            if len(parts) == 2 and parts[1] not in (".", ""):
                raw = float(parts[1]) / 100
                log.info("Terminal growth (FRED T10YIE): %.2f%%", raw * 100)
                return float(np.clip(raw, 0.015, 0.04))
    except Exception as exc:
        log.warning("Could not fetch FRED T10YIE: %s — using 2.5 %%", exc)
    return 0.025


def _compute_peg_thresholds(symbols: list[str]) -> tuple[float, float]:
    """25th / 75th percentile PEG across the universe. Falls back to (1.0, 2.0)."""
    pegs = []
    for sym in symbols:
        try:
            info = _safe_info(yf.Ticker(sym))
            if not info:
                continue
            pe = info.get("trailingPE")
            g  = info.get("earningsGrowth") or info.get("revenueGrowth")
            if pe and g and pe > 0 and g > 0:
                pegs.append(pe / (g * 100))
        except Exception:
            continue
    if len(pegs) < 10:
        log.warning("Too few PEG samples; using defaults (1.0, 2.0)")
        return 1.0, 2.0
    arr = np.array(pegs)
    buy, sell = float(np.percentile(arr, 25)), float(np.percentile(arr, 75))
    log.info("PEG thresholds — Buy < %.2f  Sell > %.2f  (n=%d)", buy, sell, len(arr))
    return buy, sell


def build_market_params(symbols: list[str]) -> MarketParams:
    log.info("Deriving market parameters from live data ...")
    rf  = _fetch_treasury_yield()
    erp = _fetch_equity_risk_premium(rf)
    dr  = float(np.clip(rf + erp, 0.05, 0.12))
    tg  = _fetch_terminal_growth()
    peg_buy, peg_sell = _compute_peg_thresholds(symbols)
    log.info("rf=%.2f%%  erp=%.2f%%  discount=%.2f%%  terminal_g=%.2f%%",
             rf*100, erp*100, dr*100, tg*100)
    return MarketParams(rf, erp, dr, tg, peg_buy, peg_sell)


# Per-stock helpers

def _current_price(ticker: yf.Ticker) -> Optional[float]:
    """Use fast_info first (no crumb needed); fall back to history."""
    try:
        price = ticker.fast_info.get("lastPrice") or ticker.fast_info.get("regularMarketPrice")
        if price is not None:
            return float(price)
    except Exception:
        pass
    try:
        hist = ticker.history(period="5d")
        if not hist.empty:
            close = hist["Close"]
            val = close.iloc[-1]
            # Handle MultiIndex DataFrame case
            if hasattr(val, "__len__"):
                val = val.iloc[0]
            return float(val)
    except Exception:
        pass
    return None


def _safe_info(ticker: yf.Ticker) -> Optional[dict]:
    """Return ticker.info only if it is a dict; otherwise return None."""
    try:
        info = ticker.info
        return info if isinstance(info, dict) else None
    except Exception:
        return None


def _annualised_volatility(ticker: yf.Ticker) -> Optional[float]:
    try:
        hist = ticker.history(period="1y")
        if hist.empty:
            return None
        close = hist["Close"]
        if hasattr(close, "columns"):
            close = close.iloc[:, 0]
        returns = close.pct_change().dropna()
        return float(returns.std() * np.sqrt(252)) if len(returns) > 20 else None
    except Exception:
        return None
def _signal(current: float, intrinsic: float, band: float) -> str:
    if current < intrinsic * (1 - band):
        return "Buy"
    if current > intrinsic * (1 + band):
        return "Sell"
    return "Hold"


def get_intrinsic_value_ddm(symbol: str, mp: MarketParams, band: float) -> Optional[ValuationResult]:
    """Gordon Growth Model: V = D1 / (r - g)"""
    try:
        ticker    = yf.Ticker(symbol)
        dividends = ticker.dividends.dropna()
        if len(dividends) < 2:
            return None
        years = (dividends.index[-1] - dividends.index[0]).days / 365.25
        if years < 1 or dividends.iloc[0] <= 0:
            return None
        cagr = (dividends.iloc[-1] / dividends.iloc[0]) ** (1 / years) - 1
        g    = min(cagr, mp.discount_rate * 0.90)
        intrinsic = dividends.iloc[-1] * (1 + g) / (mp.discount_rate - g)
        price = _current_price(ticker)
        if price is None or intrinsic <= 0:
            return None
        return _signal(price, intrinsic, band), intrinsic
    except Exception as exc:
        log.error("DDM failed for %s: %s", symbol, exc)
        return None


# Valuation model 2 — Graham Number

def get_intrinsic_value_graham(symbol: str, mp: MarketParams, band: float) -> Optional[ValuationResult]:
    """Graham Number: V = sqrt(22.5 x EPS x BVPS)"""
    try:
        ticker = yf.Ticker(symbol)
        info   = _safe_info(ticker)
        if info is None:
            return None
        eps    = info.get("trailingEps")
        bvps   = info.get("bookValue")
        if not eps or not bvps or eps <= 0 or bvps <= 0:
            return None
        intrinsic = np.sqrt(GRAHAM_MULTIPLIER * eps * bvps)
        price = _current_price(ticker)
        if price is None:
            return None
        return _signal(price, intrinsic, band), intrinsic
    except Exception as exc:
        log.error("Graham failed for %s: %s", symbol, exc)
        return None


# Valuation model 4 — Discounted Cash Flow

def get_intrinsic_value_dcf(symbol: str, mp: MarketParams, band: float) -> Optional[ValuationResult]:
    """Two-stage DCF on free cash flow (operating CF + capex)."""
    try:
        ticker = yf.Ticker(symbol)
        cf     = ticker.cashflow
        if cf is None or cf.empty:
            return None

        def _find_row(df, candidates):
            for label in candidates:
                matches = [i for i in df.index if label.lower() in i.lower()]
                if matches:
                    return df.loc[matches[0]]
            return None

        op_cf_row = _find_row(cf, ["Operating Cash Flow", "Total Cash From Operating"])
        capex_row = _find_row(cf, ["Capital Expenditure", "Capital Expenditures"])
        if op_cf_row is None or capex_row is None:
            return None

        op_cf = op_cf_row.dropna().iloc[:3].values.astype(float)
        capex = capex_row.dropna().iloc[:3].values.astype(float)
        if len(op_cf) < 2 or len(capex) < 2:
            return None

        fcf = op_cf + capex
        if fcf[-1] <= 0:
            return None

        growth = float(np.clip(
            (fcf[-1] / fcf[0]) ** (1 / (len(fcf) - 1)) - 1 if fcf[0] > 0 else 0.0,
            0.0, 0.30,
        ))

        # Use CAPM beta inline (RIM's _cost_of_equity helper was removed)
        try:
            stock  = yf.Ticker(symbol).history(period="5y")["Close"].pct_change().dropna()
            market = yf.Ticker("^GSPC").history(period="5y")["Close"].pct_change().dropna()
            aligned = pd.concat([stock, market], axis=1).dropna()
            beta = aligned.iloc[:, 0].cov(aligned.iloc[:, 1]) / aligned.iloc[:, 1].var() if not aligned.empty else 1.0
            ke = mp.risk_free_rate + beta * mp.equity_risk_premium
        except Exception:
            ke = mp.discount_rate
        if ke <= mp.terminal_growth:
            ke = mp.discount_rate

        pv_fcf = sum(fcf[-1] * (1 + growth) ** t / (1 + ke) ** t
                     for t in range(1, DCF_FORECAST_YEARS + 1))
        terminal_fcf = fcf[-1] * (1 + growth) ** DCF_FORECAST_YEARS * (1 + mp.terminal_growth)
        pv_terminal  = (terminal_fcf / (ke - mp.terminal_growth)) / (1 + ke) ** DCF_FORECAST_YEARS
        total_value  = pv_fcf + pv_terminal

        _info  = _safe_info(ticker)
        shares = _info.get("sharesOutstanding") if _info else None
        if not shares or shares <= 0:
            return None
        intrinsic = total_value / shares
        if intrinsic <= 0:
            return None
        price = _current_price(ticker)
        if price is None:
            return None
        return _signal(price, intrinsic, band), intrinsic
    except Exception as exc:
        log.error("DCF failed for %s: %s", symbol, exc)
        return None


# Valuation model 5 — PEG Ratio

def get_intrinsic_value_peg(symbol: str, mp: MarketParams, band: float) -> Optional[ValuationResult]:
    """PEG = (P/E) / EPS_growth_%%.  Buy/Sell thresholds are universe percentiles."""
    try:
        ticker = yf.Ticker(symbol)
        info   = _safe_info(ticker)
        if info is None:
            return None
        pe     = info.get("trailingPE")
        eps    = info.get("trailingEps")
        if pe is None or eps is None or pe <= 0 or eps <= 0:
            return None

        growth_rate = info.get("earningsGrowth") or info.get("revenueGrowth")
        if growth_rate is None or np.isnan(growth_rate) or growth_rate <= 0:
            income = ticker.income_stmt
            if income is not None and not income.empty:
                ni_row = next(
                    (income.loc[i] for i in income.index if "net income" in i.lower()), None
                )
                shares = info.get("sharesOutstanding") if isinstance(info, dict) else None
                if ni_row is not None and shares and shares > 0:
                    ni = ni_row.dropna().values.astype(float)
                    if len(ni) >= 2 and ni[-1] > 0 and ni[0] > 0:
                        growth_rate = (ni[0] / ni[-1]) ** (1 / (len(ni) - 1)) - 1
            if growth_rate is None or growth_rate <= 0:
                return None

        growth_pct = growth_rate * 100
        if growth_pct <= 0:
            return None

        peg = pe / growth_pct
        signal = (
            "Buy"  if peg < mp.peg_buy_threshold  else
            "Sell" if peg > mp.peg_sell_threshold  else
            "Hold"
        )
        return signal, eps * growth_pct
    except Exception as exc:
        log.error("PEG failed for %s: %s", symbol, exc)
        return None


# Aggregation

VALUATION_FNS = {
    "ddm":    get_intrinsic_value_ddm,
    "graham": get_intrinsic_value_graham,
    "dcf":    get_intrinsic_value_dcf,
    "peg":    get_intrinsic_value_peg,
}


def majority_vote(results: dict) -> str:
    signals = [r[0] for r in results.values() if r is not None]
    if not signals:
        return "Insufficient data"
    counts    = pd.Series(signals).value_counts()
    top_count = counts.iloc[0]
    top       = counts[counts == top_count].index.tolist()
    if len(top) == 1:
        return top[0]
    for preferred in ("Hold", "Buy", "Sell"):
        if preferred in top:
            return preferred
    return top[0]


def consensus_price(results: dict) -> Optional[float]:
    values = [r[1] for r in results.values() if r is not None]
    return float(np.median(values)) if values else None


def _analyze_one(symbol: str, mp: MarketParams) -> dict:
    try:
        ticker = yf.Ticker(symbol)
        price  = _current_price(ticker)
        vol    = _annualised_volatility(ticker)
        band   = float(np.clip(vol, 0.10, 0.40)) if vol is not None else 0.20
        results = {name: fn(symbol, mp, band) for name, fn in VALUATION_FNS.items()}
        return {
            "symbol":        symbol,
            "current_price": price,
            "model_price":   consensus_price(results),
            "hold_band":     band,
            **{name: (r[0] if r else None) for name, r in results.items()},
            "verdict":       majority_vote(results),
        }
    except Exception as exc:
        log.error("Unhandled error analysing %s: %s", symbol, exc)
        return {
            "symbol": symbol, "current_price": None, "model_price": None,
            "hold_band": 0.20, "ddm": None, "graham": None,
            "dcf": None, "peg": None, "verdict": "Error",
        }


def analyze_stocks(symbols: list[str], mp: MarketParams, delay: float = 1.0) -> pd.DataFrame:
    """
    Analyse stocks sequentially with a fixed delay between each one.
    Yahoo Finance's free tier rate-limits concurrent requests aggressively;
    sequential access with a small sleep is more reliable than parallelism.

    delay: seconds to sleep between stocks (default 1.0).
           Lower to 0.5 if you have a paid Yahoo Finance plan or see no errors.
           Raise to 2.0 if you still see rate-limit errors.
    """
    total = len(symbols)
    log.info("Analysing %d stocks sequentially (%.1fs delay) ...", total, delay)
    rows = []
    for i, sym in enumerate(symbols, 1):
        rows.append(_analyze_one(sym, mp))
        if i % 25 == 0 or i == total:
            log.info("Progress: %d / %d", i, total)
        if i < total:
            time.sleep(delay)
    return pd.DataFrame(rows).sort_values("symbol").set_index("symbol")


def format_results(df: pd.DataFrame, mp: MarketParams, universe_name: str) -> str:
    col_width = max(len(str(s)) for s in df.index) + 2
    header = (
        f"{'Symbol':<{col_width}}"
        f"{'DDM':>8}  {'Graham':>8}  {'DCF':>8}  {'PEG':>8}  "
        f"{'Current':>10}  {'Model':>10}  {'Band':>6}  {'Verdict':>12}"
    )
    sep  = "=" * len(header)
    thin = "-" * len(header)

    def fmt(v):
        return f"{'—':>8}" if v is None or (isinstance(v, float) and np.isnan(v)) else f"{v:>8}"

    lines = [
        sep,
        f"  {universe_name}  ({len(df)} stocks)",
        f"  rf={mp.risk_free_rate*100:.2f}%  erp={mp.equity_risk_premium*100:.2f}%  "
        f"discount={mp.discount_rate*100:.2f}%  terminal_g={mp.terminal_growth*100:.2f}%  "
        f"PEG buy<{mp.peg_buy_threshold:.2f} sell>{mp.peg_sell_threshold:.2f}",
        sep, header, thin,
    ]
    for symbol, row in df.iterrows():
        current = f"${row['current_price']:.2f}" if pd.notna(row["current_price"]) else "N/A"
        model   = f"${row['model_price']:.2f}"   if pd.notna(row["model_price"])   else "N/A"
        lines.append(
            f"{str(symbol):<{col_width}}"
            f"{fmt(row['ddm'])}  {fmt(row['graham'])}  "
            f"{fmt(row['dcf'])}  {fmt(row['peg'])}  "
            f"{current:>10}  {model:>10}  {row['hold_band']*100:.0f}%  {row['verdict']:>12}"
        )
    lines += [thin, "  " + "  ".join(f"{k}: {v}" for k, v in df['verdict'].value_counts().items()), sep]
    return "\n".join(lines)

---
## S&P 100
~100 stocks · estimated runtime ~5 min with sequential, ~1s/stock.

In [3]:
UNIVERSE_NAME = "S&P 100"

# Symbol fetcher

def fetch_symbols() -> list[str]:
    """S&P 100 constituents from Wikipedia."""
    try:
        tables = pd.read_html(
            "https://en.wikipedia.org/wiki/S%26P_100",
            storage_options={"User-Agent": "Mozilla/5.0"},
        )
        for table in tables:
            if "Symbol" in table.columns:
                syms = [s.replace(".", "-") for s in table["Symbol"].tolist()]
                log.info("Fetched %d S&P 100 symbols", len(syms))
                return syms
    except Exception as exc:
        log.error("Failed to fetch S&P 100 symbols: %s", exc)
    return []

if __name__ == "__main__":
    # ── Install dependencies if running in Colab ──────────────────────────────
    # Uncomment the line below the first time you run this notebook:
    # !pip install yfinance pandas numpy lxml html5lib beautifulsoup4 -q

    symbols = fetch_symbols()
    if not symbols:
        raise SystemExit("No symbols fetched — check your internet connection.")

    mp      = build_market_params(symbols)
    results = analyze_stocks(symbols, mp)

    output  = format_results(results, mp, UNIVERSE_NAME)
    print()
    print(output)
    print()
    print(results["verdict"].value_counts().to_string())


  S&P 100  (101 stocks)
  rf=4.50%  erp=4.20%  discount=8.70%  terminal_g=2.50%  PEG buy<0.49 sell>2.97
Symbol      DDM    Graham       DCF       PEG     Current       Model    Band       Verdict
-------------------------------------------------------------------------------------------
AAPL       Sell      Sell      Sell      Hold     $251.94      $57.92  32%          Sell
ABBV       Hold         —       Buy      Sell     $204.72     $214.42  27%          Hold
ABT        Sell      Sell      Sell         —     $104.71      $56.61  23%          Sell
ACN        Hold      Sell      Hold      Sell     $201.63     $150.50  33%          Hold
ADBE       Sell      Sell      Hold      Hold     $248.12     $147.19  31%          Hold
AMAT       Sell      Sell      Sell      Hold     $364.70     $164.58  40%          Sell
AMD           —      Sell      Sell       Buy     $204.51      $47.73  40%          Sell
AMGN       Hold      Sell      Hold       Buy     $351.39     $302.26  29%          Hold

---
## S&P 500
~500 stocks · estimated runtime ~25 min with sequential, ~1s/stock.

In [4]:
UNIVERSE_NAME = "S&P 500"


# Symbol fetcher

def fetch_symbols() -> list[str]:
    """S&P 500 constituents from Wikipedia."""
    try:
        tables = pd.read_html(
            "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
            storage_options={"User-Agent": "Mozilla/5.0"},
        )
        for table in tables:
            if "Symbol" in table.columns:
                syms = [s.replace(".", "-") for s in table["Symbol"].tolist()]
                log.info("Fetched %d S&P 500 symbols", len(syms))
                return syms
    except Exception as exc:
        log.error("Failed to fetch S&P 500 symbols: %s", exc)
    return []

if __name__ == "__main__":
    # ── Install dependencies if running in Colab ──────────────────────────────
    # Uncomment the line below the first time you run this notebook:
    # !pip install yfinance pandas numpy lxml html5lib beautifulsoup4 -q

    symbols = fetch_symbols()
    if not symbols:
        raise SystemExit("No symbols fetched — check your internet connection.")

    mp      = build_market_params(symbols)
    results = analyze_stocks(symbols, mp)

    output  = format_results(results, mp, UNIVERSE_NAME)
    print()
    print(output)
    print()
    print(results["verdict"].value_counts().to_string())

ERROR:__main__:DCF failed for ABNB: operands could not be broadcast together with shapes (3,) (2,) 
ERROR:__main__:DCF failed for APO: operands could not be broadcast together with shapes (3,) (2,) 
ERROR:__main__:DCF failed for TFC: operands could not be broadcast together with shapes (3,) (2,) 



  S&P 500  (503 stocks)
  rf=4.50%  erp=4.20%  discount=8.70%  terminal_g=2.50%  PEG buy<0.48 sell>2.64
Symbol      DDM    Graham       DCF       PEG     Current       Model    Band       Verdict
-------------------------------------------------------------------------------------------
A          Sell      Sell       Buy      Sell     $112.64      $27.87  32%          Sell
AAPL       Sell      Sell      Sell      Hold     $251.87      $57.92  32%          Sell
ABBV       Hold         —       Buy      Sell     $204.80     $214.42  27%          Hold
ABNB          —      Sell         —      Sell     $132.66      $37.54  35%          Sell
ABT        Sell      Sell      Sell         —     $104.73      $56.60  23%          Sell
ACGL          —       Buy       Buy       Buy      $93.90     $310.82  24%           Buy
ACN        Hold      Sell      Hold      Sell     $200.97     $150.50  33%          Hold
ADBE       Sell      Sell      Hold      Hold     $247.66     $147.19  31%          Hold

---
## S&P 400 Mid Cap
~400 stocks · estimated runtime ~20 min with sequential, ~1s/stock.

In [5]:
UNIVERSE_NAME = "S&P 400 Mid Cap"


# Symbol fetcher

def fetch_symbols() -> list[str]:
    """S&P 400 Mid Cap constituents from Wikipedia."""
    try:
        tables = pd.read_html(
            "https://en.wikipedia.org/wiki/List_of_S%26P_400_companies",
            storage_options={"User-Agent": "Mozilla/5.0"},
        )
        for table in tables:
            if "Symbol" in table.columns:
                syms = [s.replace(".", "-") for s in table["Symbol"].tolist()]
                log.info("Fetched %d S&P 400 Mid Cap symbols", len(syms))
                return syms
    except Exception as exc:
        log.error("Failed to fetch S&P 400 Mid Cap symbols: %s", exc)
    return []

if __name__ == "__main__":
    # ── Install dependencies if running in Colab ──────────────────────────────
    # Uncomment the line below the first time you run this notebook:
    # !pip install yfinance pandas numpy lxml html5lib beautifulsoup4 -q

    symbols = fetch_symbols()
    if not symbols:
        raise SystemExit("No symbols fetched — check your internet connection.")

    mp      = build_market_params(symbols)
    results = analyze_stocks(symbols, mp)

    output  = format_results(results, mp, UNIVERSE_NAME)
    print()
    print(output)
    print()
    print(results["verdict"].value_counts().to_string())

ERROR:__main__:DCF failed for RGA: operands could not be broadcast together with shapes (3,) (2,) 
ERROR:__main__:DCF failed for VNO: operands could not be broadcast together with shapes (3,) (2,) 



  S&P 400 Mid Cap  (400 stocks)
  rf=4.50%  erp=4.20%  discount=8.70%  terminal_g=2.50%  PEG buy<0.30 sell>2.76
Symbol     DDM    Graham       DCF       PEG     Current       Model    Band       Verdict
------------------------------------------------------------------------------------------
AA        Sell      Hold         —      Hold      $56.50      $27.97  40%          Hold
AAL       Sell         —      Hold         —      $10.86       $9.48  40%          Hold
AAON      Sell      Sell      Sell      Sell      $80.70       $4.62  40%          Sell
ACI       Hold      Sell       Buy         —      $17.00      $18.59  30%          Hold
ACM       Sell      Sell      Sell      Hold      $90.58      $53.79  30%          Sell
ADC       Sell      Sell         —      Sell      $76.16      $24.78  17%          Sell
AEIS      Sell      Sell      Sell      Sell     $331.27      $29.95  40%          Sell
AFG        Buy      Hold         —      Hold     $128.09     $183.46  23%          Hold
A

---
## Russell 2000
~2000 stocks · estimated runtime ~90 min with sequential, ~1s/stock.

In [6]:
UNIVERSE_NAME = "Russell 2000"


# Symbol fetcher

def fetch_symbols() -> list[str]:
    """Russell 2000 constituents from the iShares IWM ETF holdings CSV."""
    url = (
        "https://www.ishares.com/us/products/239710/ishares-russell-2000-etf"
        "/1467271812596.ajax?fileType=csv&fileName=IWM_holdings&dataType=fund"
    )
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=30) as resp:
            raw = resp.read().decode("utf-8", errors="replace")
        lines = raw.splitlines()
        data_start = next(
            (i for i, l in enumerate(lines) if l.startswith("Ticker,")), None
        )
        if data_start is None:
            raise ValueError("Could not find Ticker header row in IWM CSV")
        df = pd.read_csv(io.StringIO("\n".join(lines[data_start:])), on_bad_lines="skip")
        df = df[df["Asset Class"] == "Equity"]
        syms = (
            df["Ticker"].dropna().astype(str)
            .str.replace(".", "-", regex=False).str.strip().tolist()
        )
        syms = [s for s in syms if s and s != "-"]
        log.info("Fetched %d Russell 2000 symbols", len(syms))
        return syms
    except Exception as exc:
        log.error("Failed to fetch Russell 2000 symbols: %s", exc)
    return []

if __name__ == "__main__":
    # ── Install dependencies if running in Colab ──────────────────────────────
    # Uncomment the line below the first time you run this notebook:
    # !pip install yfinance pandas numpy lxml html5lib beautifulsoup4 -q

    symbols = fetch_symbols()
    if not symbols:
        raise SystemExit("No symbols fetched — check your internet connection.")

    mp      = build_market_params(symbols)
    results = analyze_stocks(symbols, mp)

    output  = format_results(results, mp, UNIVERSE_NAME)
    print()
    print(output)
    print()
    print(results["verdict"].value_counts().to_string())

ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: GEFB"}}}
ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: P5N994"}}}
ERROR:yfinance:$MOGA: possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")
ERROR:yfinance:$MOGA: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")
ERROR:yfinance:$MOGA: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")
ERROR:yfinance:$MOGA: possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")
ERROR:yfinance:$MOGA: possibly delisted; no timezone found
ERROR:__main__:PEG failed for ALHC: '<=' not supported between instances of 'str' and 'int'
ERROR:__main__:DCF failed for SYRE: operand


  Russell 2000  (1939 stocks)
  rf=4.50%  erp=4.20%  discount=8.70%  terminal_g=2.50%  PEG buy<0.21 sell>1.92
Symbol       DDM    Graham       DCF       PEG     Current       Model    Band       Verdict
--------------------------------------------------------------------------------------------
AAMI        Sell      Sell      Sell         —      $52.13       $9.19  40%          Sell
AAOI           —         —         —         —      $95.76         N/A  40%  Insufficient data
AAP         Sell      Sell      Sell         —      $50.74      $21.89  40%          Sell
AARD           —         —         —         —       $4.07         N/A  40%  Insufficient data
AAT         Sell      Hold       Buy      Sell      $18.73      $14.34  26%          Sell
ABAT           —         —         —         —       $2.90         N/A  40%  Insufficient data
ABCB        Sell      Hold       Buy      Hold      $75.76      $95.75  29%          Hold
ABEO           —       Buy         —         —       $4.42